In [ ]:
from IPython.display import clear_output
!git clone --branch shantanu https://github.com/AISC-Linear-Probe-Gen/Probe-Generalisation.git
%pip install --upgrade mech-interp-toolkit
clear_output()
import os
os.chdir("/content/Probe-Generalisation/research/obfuscated_activations")

In [ ]:
import queue
import threading
import warnings
from typing import cast

import einops
import torch
from datasets import load_dataset
from google.colab import userdata
from huggingface_hub import HfApi, login
from pathlib import Path
from tqdm import tqdm

from utils.data import extract_user_instruction
from mech_interp_toolkit.utils import load_model_tokenizer_config, set_global_seed
from mech_interp_toolkit.activation_utils import get_embeddings_dict, get_activations

warnings.filterwarnings("ignore")
set_global_seed(0)
torch.set_grad_enabled(False)

In [ ]:
dataset_name = "Mechanistic-Anomaly-Detection/llama3-jailbreaks"
split = "circuit_breakers_test"
model_name = "meta-llama/Llama-3.2-3B-Instruct"
suffix_paths = ["ra_suffix.pt", "sd_suffix.pt"]          # list of .pt embedding files
save_folder_names = ["obfuscated_behavioral", "behavioral"]    # corresponding folder names under outputs/cached_activations/
batch_size = 32
compute_baseline = True               # set to False to skip baseline activation computation and saving

assert len(suffix_paths) == len(save_folder_names), \
    "suffix_paths and save_folder_names must have the same length"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
dataset = load_dataset(dataset_name, split=split)
prompts_str = [extract_user_instruction(x) for x in dataset["prompt"]]
clear_output()

model, ch_tokenizer, config = load_model_tokenizer_config(
    model_name,
    suffix="",
    system_prompt="",
    attn_type="sdpa",
)
clear_output()


def load_suffix(suffix_path: str, device: torch.device) -> torch.Tensor:
    if not suffix_path.endswith(".pt"):
        raise ValueError("suffix_path must be a .pt embedding file")
    suffix_emb = torch.load(suffix_path, map_location=device)
    if suffix_emb.dim() == 2:
        suffix_emb = suffix_emb.unsqueeze(0)
    return suffix_emb

In [ ]:
components = [(i, "layer_out") for i in range(config.num_hidden_layers)]

login(token=userdata.get('HF_TOKEN'))
api = HfApi()
repo_id = "AISC-Linear-Probe-Gen/obfuscated_activations"
api.create_repo(repo_id=repo_id, repo_type="dataset", exist_ok=True)

tmp_dir = Path("outputs/tmp_acts")
tmp_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
import time

_STOP = object()
_MAX_RETRIES = 5
_RETRY_BASE_DELAY = 5  # seconds; doubles on each retry

# Shared failure flag: worker sets it so the main thread can raise after joining
_upload_failed = threading.Event()


def _upload_worker(q: queue.Queue) -> None:
    """Save each tensor to a temp file, upload to HF with retries, then free both."""
    while True:
        item = q.get()
        if item is _STOP:
            q.task_done()
            break
        tensor, tmp_path, path_in_repo = item
        try:
            torch.save(tensor, tmp_path)
            del tensor  # free RAM before the (potentially slow) upload

            for attempt in range(1, _MAX_RETRIES + 1):
                try:
                    api.upload_file(
                        path_or_fileobj=str(tmp_path),
                        path_in_repo=path_in_repo,
                        repo_id=repo_id,
                        repo_type="dataset",
                    )
                    tmp_path.unlink()
                    break  # success
                except Exception as e:
                    if attempt == _MAX_RETRIES:
                        print(f"[upload worker] FAILED after {_MAX_RETRIES} attempts: {path_in_repo}\n  {e}")
                        print(f"[upload worker] Keeping temp file at: {tmp_path}")
                        _upload_failed.set()
                    else:
                        delay = _RETRY_BASE_DELAY * (2 ** (attempt - 1))
                        print(f"[upload worker] attempt {attempt} failed for {path_in_repo}, retrying in {delay}s...\n  {e}")
                        time.sleep(delay)
        except Exception as e:
            print(f"[upload worker] ERROR saving tensor for {path_in_repo}: {e}")
            _upload_failed.set()
        finally:
            q.task_done()


upload_queue: queue.Queue = queue.Queue()
upload_thread = threading.Thread(target=_upload_worker, args=(upload_queue,), daemon=True)
upload_thread.start()

try:
    # Baseline pass: computed once (independent of any suffix)
    if compute_baseline:
        print("\n=== Computing baseline activations ===")
        for batch_start in tqdm(range(0, len(prompts_str), batch_size), desc="baseline"):
            batch_prompts = prompts_str[batch_start : batch_start + batch_size]

            batch_dict = ch_tokenizer(prompts=batch_prompts)
            batch_embeds_dict = get_embeddings_dict(model, batch_dict)

            batch_idx = batch_start // batch_size
            filename = f"{batch_idx}_{split}_all_pos.pt"

            base_acts = get_activations(
                model, inputs=batch_embeds_dict, layer_components=components,
                retain_grads=False, positions=None,
            ).cpu()
            upload_queue.put((base_acts, tmp_dir / f"base_{filename}", f"baseline/{filename}"))

    for suffix_path, folder_name in zip(suffix_paths, save_folder_names):
        print(f"\n=== Processing suffix: {suffix_path} -> {folder_name} ===")

        suffix_embed = load_suffix(suffix_path, device)
        len_suffix = suffix_embed.shape[1]

        for batch_start in tqdm(range(0, len(prompts_str), batch_size), desc=folder_name):
            batch_prompts = prompts_str[batch_start : batch_start + batch_size]

            batch_dict = ch_tokenizer(prompts=batch_prompts)
            batch_embeds_dict = get_embeddings_dict(model, batch_dict)
            batch_embeds = batch_embeds_dict["inputs_embeds"]
            batch_attn_mask = batch_embeds_dict["attention_mask"]

            batch_suffix = einops.repeat(
                suffix_embed,
                "1 pos d_model -> b pos d_model",
                b=len(batch_prompts),
            )
            new_embeds_dict = {
                "inputs_embeds": torch.cat(
                    [batch_embeds[:, :-5, :], batch_suffix, batch_embeds[:, -5:, :]], dim=1
                ),
                "attention_mask": torch.cat(
                    [batch_attn_mask, torch.ones(len(batch_prompts), len_suffix, device=batch_attn_mask.device)],
                    dim=1,
                ),
            }

            batch_idx = batch_start // batch_size
            filename = f"{batch_idx}_{split}_all_pos.pt"

            new_acts = get_activations(
                model, inputs=new_embeds_dict, layer_components=components,
                retain_grads=False, positions=None,
            ).cpu()
            upload_queue.put((new_acts, tmp_dir / f"obf_{filename}", f"{folder_name}/{filename}"))

finally:
    upload_queue.put(_STOP)
    upload_thread.join()

if _upload_failed.is_set():
    raise RuntimeError("One or more uploads failed permanently. Check logs above for which files were kept.")

print("\nDone — all activations uploaded to HuggingFace.")
